# Disk-Resident Dataset Analysis

This notebook performs consistency checks over the dataset registry and the artifact contracts used by the five disk-resident baselines. It mirrors the role of `memory_resident_ann/data/dataset_analysis.ipynb`, but the analysis focus here is registry completeness, static/dynamic artifact coverage, and baseline-specific coupling constraints.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd

DATA_DIR = Path.cwd()
with (DATA_DIR / "config.yml").open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

dataset_manifest = cfg["dataset"]["manifest"]
dataset_groups = cfg["dataset"]["groups"]
baselines = cfg["baselines"]


## Dataset Field Completeness

Each dataset entry must expose the minimum fields needed by the disk-resident build paths. Missing fields at this layer would break every downstream baseline-specific import stage.

In [ ]:
required_dataset_fields = {
    "paper_label",
    "group",
    "value_type",
    "distance",
    "scale",
    "base_file",
    "query_file",
    "groundtruth_file",
    "note",
}

rows = []
for name, meta in dataset_manifest.items():
    missing = sorted(required_dataset_fields - set(meta))
    rows.append({
        "dataset": name,
        "group": meta.get("group", ""),
        "scale": meta.get("scale", ""),
        "missing_fields": ", ".join(missing),
        "ok": not missing,
    })
pd.DataFrame(rows)

## Group Coverage Check

This section verifies that every prepared dataset belongs to one declared group and that the registry still keeps the original paper datasets separate from the four 100M-scale additions.

In [ ]:
prepare_set = set(cfg["dataset"]["prepare"])
rows = []
for group_name, meta in dataset_groups.items():
    group_set = set(meta["datasets"])
    rows.append({
        "group": group_name,
        "declared_count": len(group_set),
        "prepared_count": len(group_set & prepare_set),
        "all_prepared": group_set <= prepare_set,
        "datasets": ", ".join(meta["datasets"]),
    })
pd.DataFrame(rows)

## Access-Profile and Window Schema Check

The static QSO layer depends on the access-profile columns, while the dynamic RDO layer depends on the workload-window fields. This cell confirms that the schema declarations remain stable and non-empty.

In [ ]:
schema_summary = pd.DataFrame([
    {
        "schema": "access_profile",
        "field_count": len(cfg["access_profile"]["columns"]),
        "fields": ", ".join(cfg["access_profile"]["columns"]),
    },
    {
        "schema": "workload_windows",
        "field_count": len(cfg["workload_windows"]["fields"]),
        "fields": ", ".join(cfg["workload_windows"]["fields"]),
    },
])
schema_summary

## Static QSO Artifact Coverage

Every baseline must declare which physical layout unit QSO controls, where the imported order enters the build path, and which rebuilt artifacts are expected as output.

In [ ]:
rows = []
for name, meta in baselines.items():
    static_meta = meta["static_qso"]
    rows.append({
        "baseline": name,
        "layout_unit": meta["layout_unit"],
        "artifact_count": len(static_meta["artifact_names"]),
        "has_import_stage": bool(static_meta["import_stage"]),
        "has_rebuild_output": bool(static_meta["rebuild_output"]),
        "artifact_names": ", ".join(static_meta["artifact_names"]),
        "rebuild_output": ", ".join(static_meta["rebuild_output"]),
    })
pd.DataFrame(rows)

## Dynamic RDO Artifact Coverage

This section checks that each baseline also carries a dynamic layout family, a switch unit, and a replay target. These fields are what make the disk-resident package look like a complete RDO integration surface instead of a static-only extension.

In [ ]:
rows = []
for name, meta in baselines.items():
    dynamic_meta = meta["dynamic_rdo"]
    rows.append({
        "baseline": name,
        "layout_family_entries": len(dynamic_meta["layout_family"]),
        "has_switch_unit": bool(dynamic_meta["switch_unit"]),
        "has_replay_target": bool(dynamic_meta["replay_target"]),
        "layout_family": ", ".join(dynamic_meta["layout_family"]),
        "switch_unit": dynamic_meta["switch_unit"],
    })
pd.DataFrame(rows)

## Baseline Data-Field Matrix

The dataset registry is shared, but each baseline consumes a different set of build-time fields. This matrix makes it easy to inspect whether the five integration notes are grounded in a concrete per-system input contract.

In [ ]:
rows = []
for name, meta in baselines.items():
    rows.append({
        "baseline": name,
        "field_count": len(meta["data_fields"]),
        "data_fields": ", ".join(meta["data_fields"]),
    })
pd.DataFrame(rows)

## Static vs Dynamic Artifact Density

A quick artifact-density view helps compare how much layout structure is carried by each baseline in the static QSO layer versus the dynamic RDO layer.

In [ ]:
rows = []
for name, meta in baselines.items():
    rows.append({
        "baseline": name,
        "static_artifacts": len(meta["static_qso"]["artifact_names"]),
        "dynamic_family_entries": len(meta["dynamic_rdo"]["layout_family"]),
        "data_fields": len(meta["data_fields"]),
    })
pd.DataFrame(rows)

## Requirement Group Consistency

The data-side package is also expected to stay aligned with the environment groups needed by the five baselines. This check ensures each baseline still has a corresponding requirement group.

In [ ]:
requirement_groups = cfg["requirements"]["system_groups"]
rows = []
for name in baselines:
    rows.append({
        "baseline": name,
        "has_requirement_group": name in requirement_groups,
        "requirement_count": len(requirement_groups.get(name, [])),
        "requirements": ", ".join(requirement_groups.get(name, [])),
    })
pd.DataFrame(rows)

## Registry Summary

The summary below provides a compact high-level count of the dataset and artifact surface represented in this directory.

In [ ]:
summary = {
    "available_datasets": len(cfg["dataset"]["available"]),
    "prepared_datasets": len(cfg["dataset"]["prepare"]),
    "dataset_groups": len(dataset_groups),
    "baselines": len(baselines),
    "static_artifacts_total": sum(len(meta["static_qso"]["artifact_names"]) for meta in baselines.values()),
    "dynamic_family_entries_total": sum(len(meta["dynamic_rdo"]["layout_family"]) for meta in baselines.values()),
}
pd.DataFrame([summary])

## Notes

The goal of this notebook is not to benchmark the baselines. It verifies that the dataset-side registry already carries the full static and dynamic artifact vocabulary needed by PageANN, SPANN, Starling, MARGO, and Gorgeous before the reader drills into the project-level integration notes.